# RNN lab: next-word prediction on Shakespeare's Sonnets
**Name:** ______________  **Student ID (NIM):** ______________

Rules: PyTorch/NumPy/Python docs are allowed. Chatbots, code assistants and copying from classmates are not.
Your configuration (below) is generated from your NIM, so your numbers will differ from your neighbour's.
Cells marked **TODO** are yours to write. Cells marked **CHECK** run without edits and tell you whether your
code is right, so you never need outside help to know if you are on track.

In [ ]:
import re, math, random, collections
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt

NIM = 0          # <-- type your student ID here (digits only)
assert NIM != 0, "Set NIM first"

In [ ]:
def make_config(nim):
    rng = random.Random(nim)
    return dict(
        seq_len    = rng.choice([10, 15, 20]),        # words the RNN reads per training window
        hidden     = rng.choice([64, 96, 128]),      # size of the hidden state
        emb        = 64,
        lr         = rng.choice([0.002, 0.003, 0.004]),
        batch      = 64,
        epochs     = 8,
        test_start = rng.randrange(0, 146),           # 8 consecutive sonnets are held out for testing
        prompt     = rng.choice([["shall", "i", "compare"], ["when", "i", "do"], ["my", "love", "is"],
                                 ["thy", "sweet", "love"], ["o", "that", "time"], ["love", "is", "not"]]),
        seed       = nim % 10000,
    )
cfg = make_config(NIM)
torch.manual_seed(cfg["seed"]); random.seed(cfg["seed"]); np.random.seed(cfg["seed"])
print(cfg)

## Step 0: Load the data (provided)
154 sonnets, each 14 lines. Headers are Roman numerals alone on a line. If GitHub is blocked, use the `sonnets.txt` from the LMS.

In [ ]:
import os, urllib.request
URL = "https://raw.githubusercontent.com/martin-gorner/tensorflow-rnn-shakespeare/master/shakespeare/sonnets.txt"
if os.path.exists("sonnets.txt"):
    raw = open("sonnets.txt", encoding="utf-8").read()
else:
    raw = urllib.request.urlopen(URL).read().decode("utf-8")
parts = re.split(r'^\s*([IVXLC]+)\.\s*$', raw, flags=re.M)
sonnets = [parts[i].strip() for i in range(2, len(parts), 2)]
print(len(sonnets), "sonnets")           # expect 154
print(sonnets[17][:120])

## Step 1: Tokenize  *(TODO 1)*
Turn text into a list of tokens: lowercase **words** (letters and apostrophes, so `beauty's` stays one token) and the
punctuation marks `, . ; : ! ?` as separate tokens. Everything else (hyphens, brackets, digits) is dropped.

In [ ]:
def tokenize(text):
    # TODO 1
    raise NotImplementedError

In [ ]:
# CHECK 1
assert tokenize("Thou art, O Love's child!") == ['thou', 'art', ',', 'o', "love's", 'child', '!'], tokenize("Thou art, O Love's child!")
assert tokenize("A-b (c)") == ['a', 'b', 'c']
print("CHECK 1 passed")

## Step 2: Split by poem, build the vocabulary  *(TODO 2)*
The test set is 8 consecutive sonnets chosen by your config. **Build the vocabulary from the training poems only.**
Keep words that appear at least twice; everything else becomes `<unk>`. Index 0 must be `<unk>` and index 1 `<eos>`.
Each poem ends with an `<eos>` token (already added for you).

In [ ]:
test_ids = set(range(cfg["test_start"], cfg["test_start"] + 8))
train_poems = [s for i, s in enumerate(sonnets) if i not in test_ids]
test_poems  = [s for i, s in enumerate(sonnets) if i in test_ids]

def poems_to_tokens(poems):
    out = []
    for p in poems:
        out += tokenize(p) + ["<eos>"]
    return out
train_tokens, test_tokens = poems_to_tokens(train_poems), poems_to_tokens(test_poems)

# TODO 2: build itos (list of words, index -> word), stoi (dict, word -> index) and V (vocabulary size)
itos, stoi, V = None, None, None

In [ ]:
# CHECK 2
assert itos[0] == "<unk>" and itos[1] == "<eos>" and stoi["<eos>"] == 1
assert all(stoi[w] == i for i, w in enumerate(itos)) and len(itos) == V
assert "love" in stoi and 1000 < V < 2000, V
print("CHECK 2 passed.  V =", V, "  <- write this number in your journal")

## Step 3: Encode and make (context, next-word) windows  *(TODO 3, TODO 4)*
For a window length `L`, input `X[i]` is the `L` word ids starting at position `i`, and target `Y[i]` is the same
window **shifted one step to the right**. So at every position the model is asked: given the words so far, what comes next?

In [ ]:
def encode(tokens):
    # TODO 3: list of tokens -> 1-D LongTensor of ids (unknown words -> 0)
    raise NotImplementedError

def make_windows(ids, L):
    # TODO 4: return X, Y, both LongTensors of shape (n, L), with n = len(ids) - L
    raise NotImplementedError

In [ ]:
# CHECK 3
t = encode(["love", "zzzzunseen", "<eos>"]); assert t.tolist() == [stoi["love"], 0, 1] and t.dtype == torch.long
ids = torch.arange(10); X, Y = make_windows(ids, 4)
assert X.shape == (6, 4) and Y.shape == (6, 4)
assert X[2].tolist() == [2, 3, 4, 5] and Y[2].tolist() == [3, 4, 5, 6]
print("CHECK 3 passed")
L = cfg["seq_len"]
Xtr, Ytr = make_windows(encode(train_tokens), L)
Xte, Yte = make_windows(encode(test_tokens), L)
print("train windows", tuple(Xtr.shape), " test windows", tuple(Xte.shape))

## Step 4: The RNN cell, written by hand  *(TODO 5, 6, 7)*
No `nn.RNN` allowed here. One step of the recurrence for a batch is

`h_t = tanh( e_t · Wxh + h_(t-1) · Whh + bh )`

where `e_t` is the embedding of the word at time `t`. After the loop, a linear layer turns **every** hidden state into
`V` scores (logits). Think about shapes before you type: write them as comments.

In [ ]:
class MyRNN(nn.Module):
    def __init__(self, V, E, H):
        super().__init__()
        self.H = H
        self.emb = nn.Embedding(V, E)
        # TODO 5: create three nn.Parameter tensors Wxh (E,H), Whh (H,H), bh (H,)
        #         use small random weights (scale about 1/sqrt(H)) and zero bias
        self.drop = nn.Dropout(0.5)       # regularisation: the corpus is tiny, so the model would memorise it
        self.out = nn.Linear(H, V)

    def forward(self, x, h=None):
        # x: (B, L) word ids.  Return logits (B, L, V) and the final hidden state (B, H)
        B, L = x.shape
        e = self.drop(self.emb(x))                        # (B, L, E)   dropout is only active in train mode
        if h is None:
            h = torch.zeros(B, self.H)
        states = []
        # TODO 6: loop over the L time steps, update h, collect it
        # TODO 7: stack the collected states into (B, L, H), apply self.drop, then self.out
        raise NotImplementedError

In [ ]:
# CHECK 4: your cell must reproduce PyTorch's nn.RNN exactly when given the same weights
m = MyRNN(50, 8, 16).eval()          # eval(): switches dropout off
ref = nn.RNN(8, 16, batch_first=True)
with torch.no_grad():
    ref.weight_ih_l0.copy_(m.Wxh.T); ref.weight_hh_l0.copy_(m.Whh.T)
    ref.bias_ih_l0.copy_(m.bh);      ref.bias_hh_l0.zero_()
x = torch.randint(0, 50, (3, 7))
logits, h_last = m(x)
ref_states, ref_h = ref(m.emb(x))
assert logits.shape == (3, 7, 50), logits.shape
assert torch.allclose(m.out(ref_states), logits, atol=1e-5) and torch.allclose(ref_h[0], h_last, atol=1e-5)
print("CHECK 4 passed: your RNN equals nn.RNN")

## Step 5: Train and evaluate  *(TODO 8, 9)*
* `run_epoch(model, X, Y, train)` returns the **average cross-entropy per predicted word** over all windows.
  When `train=True`: shuffle, use mini-batches of `cfg["batch"]`, backpropagate, **clip the gradient norm to 1.0**, step the optimizer.
  When `train=False`: no gradients, no updates.
* `F.cross_entropy` wants logits `(N, V)` and targets `(N,)`. Your logits are `(B, L, V)`.
* Perplexity = `exp(loss)`: roughly "how many words the model is torn between".

In [ ]:
model = MyRNN(V, cfg["emb"], cfg["hidden"])
opt = torch.optim.Adam(model.parameters(), lr=cfg["lr"])

def run_epoch(model, X, Y, train):
    # TODO 8
    raise NotImplementedError

In [ ]:
# CHECK 5 (untrained model): the loss should be close to ln(V). Write both numbers in your journal BEFORE training.
l0 = run_epoch(model, Xte, Yte, train=False)
print(f"untrained test loss = {l0:.3f}    ln(V) = {math.log(V):.3f}")
assert abs(l0 - math.log(V)) < 0.7, "A fresh model should be roughly uniform. Check your loss/reshape."

In [ ]:
import copy
tr_hist, te_hist = [], []
best, best_state = 1e9, None
for ep in range(1, cfg["epochs"] + 1):
    tr = run_epoch(model, Xtr, Ytr, True)
    te = run_epoch(model, Xte, Yte, False)
    tr_hist.append(tr); te_hist.append(te)
    if te < best:                                   # remember the best weights (early stopping)
        best, best_state = te, copy.deepcopy(model.state_dict())
    print(f"epoch {ep:2d}  train ppl {math.exp(tr):8.1f}   test ppl {math.exp(te):8.1f}")
model.load_state_dict(best_state)
plt.plot([math.exp(v) for v in tr_hist], label="train"); plt.plot([math.exp(v) for v in te_hist], label="test")
plt.xlabel("epoch"); plt.ylabel("perplexity"); plt.legend(); plt.show()

### Baseline  *(TODO 9)*
A model that ignores context and only knows word frequencies is the **unigram baseline**. Using add-one smoothing,
`p(w) = (count_train(w) + 1) / (N_train + V)` where counts are over **encoded training ids** (so `<unk>` has a count).
Compute its perplexity on the same test targets `Yte`. Your RNN must beat this number to be called a language model.

In [ ]:
# TODO 9: unigram_ppl
unigram_ppl = None

In [ ]:
print(f"unigram baseline test ppl = {unigram_ppl:.1f}")
print(f"your RNN best test ppl    = {min(math.exp(v) for v in te_hist):.1f}  (epoch {int(np.argmin(te_hist))+1})")

## Step 6: Generate a poem  *(TODO 10)*
Feed the prompt words through the model, keep the final hidden state, then repeat: turn the last logits into a probability
distribution **using the temperature**, never allow `<unk>` (give it a logit of -1e9), sample one word with `torch.multinomial`, feed *only that word* back in together
with the hidden state. `<eos>` should be printed as a line break.

In [ ]:
@torch.no_grad()
def generate(model, prompt_words, n_words=60, temperature=1.0):
    model.eval()
    # TODO 10
    raise NotImplementedError

In [ ]:
def detok(words):
    s = ""
    for w in words:
        if w == "<eos>": s += "\n\n"
        elif w in ",.;:!?": s = s.rstrip(" ") + w + " "
        else: s += w + " "
    return s

print(detok(generate(model, cfg["prompt"], 60, temperature=1.0)))

## Part C: experiments (use your paper journal!)
**Write your prediction on the journal sheet before you run each cell.**

**C1. Temperature.** Generate from your prompt with temperature 0.5, 1.0 and 1.5 (same code, change one number).
**C2. When to stop.** From the curves above, which epoch had the lowest test perplexity? Is it the last one?
**C3 (optional, if time remains).** Replace your cell by `nn.LSTM` (or `nn.RNN`) with the same sizes, retrain, compare test perplexity.

In [ ]:
for T in (0.5, 1.0, 1.5):
    print(f"--- temperature {T} ---")
    print(detok(generate(model, cfg["prompt"], 50, temperature=T)))

## Part E (optional, next session): add attention
In the RNN, `h_t` must carry everything read so far. **Attention** lets position `t` look directly at the hidden
states of all earlier positions and take a weighted average of them. You will add it on top of your RNN and compare.

Shapes to keep in mind (`B` batch, `L` window, `H` hidden): `H_all` is `(B, L, H)` -> `scores` is `(B, L, L)` -> `attn` is `(B, L, L)` -> `ctx` is `(B, L, H)`.
**Write your prediction on the journal sheet before running the training cell:** will attention lower the best test perplexity? Why or why not?

In [ ]:
rnn_model, rnn_best_ppl = model, math.exp(min(te_hist))      # keep the plain RNN for comparison

In [ ]:
class MyAttnRNN(nn.Module):
    def __init__(self, V, E, H):
        super().__init__()
        self.H = H
        self.emb = nn.Embedding(V, E)
        s = 1.0 / math.sqrt(H)
        self.Wxh = nn.Parameter(torch.randn(E, H) * s)
        self.Whh = nn.Parameter(torch.randn(H, H) * s)
        self.bh  = nn.Parameter(torch.zeros(H))
        self.drop = nn.Dropout(0.5)
        self.Wq = nn.Linear(H, H, bias=False)
        self.Wk = nn.Linear(H, H, bias=False)
        self.out = nn.Linear(2 * H, V)             # input is [H_all, ctx] side by side, so 2*H

    def forward(self, x, h=None):
        B, L = x.shape
        e = self.drop(self.emb(x))
        if h is None:
            h = torch.zeros(B, self.H)
        states = []
        for t in range(L):                          # the recurrence from Step 4 (given)
            h = torch.tanh(e[:, t] @ self.Wxh + h @ self.Whh + self.bh)
            states.append(h)
        H_all = torch.stack(states, dim=1)          # (B, L, H)
        # TODO 11a: queries and keys from H_all; scores = q k^T / sqrt(H)            -> (B, L, L)
        #           causal mask: position t may NOT look at positions s > t (set them to -inf)
        #           attn = softmax over the last dimension                            -> (B, L, L)
        # TODO 11b: ctx = attn @ H_all                                                -> (B, L, H)
        #           store self.last_attn = attn.detach(); return logits from [H_all, ctx] and h
        raise NotImplementedError

In [ ]:
# CHECK 6: attention rows sum to 1, and nothing is allowed above the diagonal (no peeking at the future)
am = MyAttnRNN(50, 8, 16).eval()
logits, _ = am(torch.randint(0, 50, (3, 7)))
A = am.last_attn
assert logits.shape == (3, 7, 50) and A.shape == (3, 7, 7), (logits.shape, A.shape)
assert torch.allclose(A.sum(-1), torch.ones(3, 7), atol=1e-5), "rows must sum to 1"
assert torch.all(torch.triu(A, diagonal=1) == 0), "future positions must get exactly 0"
assert torch.allclose(A[:, 0, 0], torch.ones(3)), "the first position can only attend to itself"
print("CHECK 6 passed")

In [ ]:
model = MyAttnRNN(V, cfg["emb"], cfg["hidden"])
opt = torch.optim.Adam(model.parameters(), lr=cfg["lr"])
a_tr, a_te = [], []
best, best_state = 1e9, None
for ep in range(1, cfg["epochs"] + 1):
    tr = run_epoch(model, Xtr, Ytr, True)
    te = run_epoch(model, Xte, Yte, False)
    a_tr.append(tr); a_te.append(te)
    if te < best:
        best, best_state = te, copy.deepcopy(model.state_dict())
    print(f"epoch {ep:2d}  train ppl {math.exp(tr):8.1f}   test ppl {math.exp(te):8.1f}")
model.load_state_dict(best_state)
print(f"plain RNN best test ppl {rnn_best_ppl:.1f}   attention best test ppl {math.exp(best):.1f}   unigram {unigram_ppl:.1f}")
print("parameters: RNN", sum(p.numel() for p in rnn_model.parameters()), " attention", sum(p.numel() for p in model.parameters()))

### Look inside  *(TODO 12 and the heatmap)*
Generation must change: the attention model needs the previous words, not just one word and a hidden state.
At every step, feed the **last `L` words generated so far** and use the logits of the last position.

In [ ]:
@torch.no_grad()
def generate_attn(model, prompt_words, n_words=60, temperature=1.0, L=cfg["seq_len"]):
    model.eval()
    # TODO 12
    raise NotImplementedError

In [ ]:
print(detok(generate_attn(model, cfg["prompt"], 60, temperature=1.0)))

# Heatmap: row = the word being read, column = the earlier word it looked at
model.eval(); model(Xte[:1])
words = [itos[i] for i in Xte[0].tolist()]
plt.figure(figsize=(6, 5)); plt.imshow(model.last_attn[0], cmap="viridis")
plt.xticks(range(len(words)), words, rotation=90); plt.yticks(range(len(words)), words)
plt.xlabel("looked at"); plt.ylabel("reading"); plt.colorbar(); plt.tight_layout(); plt.show()